# 41a · Ba CNN trong **một lượt quét** — và `corr` với RAD-DINO là con số quyết định

## Đây là một **cổng**, không phải một cải tiến

Ensemble chỉ mua được khi các thành viên **không tương quan**. `17` đo ensemble ba checkpoint của
**cùng một** mô hình: `macro +0.00112` (CI loại 0), `tail +0.00465` — nhỏ, vì tương quan cực cao.
Đa dạng **kiến trúc** phải cho nhiều hơn, nhưng chỉ khi nó thật sự đa dạng.

```
ca ba CNN co corr >= moc SAME-BACKBONE  ->  kien truc CNN KHONG mang lai da dang
                                            -> DONG huong, khoi fine-tune con nao, tiet kiem ca buoi
```

1.5 giờ để có thể xoá một hướng khỏi lịch 14 ngày.

## Mốc `corr` lấy từ **dữ liệu**, không phải phỏng đoán

`49` đã probe **bốn** arm ảnh trên **cùng** RAD-DINO pretrain (`crop518`/`lb518`/`nosq518`/`lb672`),
3 seed, cùng đầu đọc. `corr` giữa chúng chính là **cận dưới của đa dạng**: hai mô hình khác nhau
nhiều nhất mà vẫn dùng chung một backbone. Nếu CNN không xuống dưới mức đó thì nó không mang lại
gì mà một arm ảnh khác không mang lại.

`0.70` là cao hay thấp — không đoán, đọc bằng mốc này.

## Ba con, một lượt quét

```
convnextv2_nano.fcmae_ft_in22k_in1k   ~13 GMAC @518   feat 640   <- RE hon DenseNet ma IN-1k cao hon ~6pp
densenet121.ra_in1k                   ~15 GMAC        feat 1024  <- moc "da duoc dung tren CXR"
convnextv2_tiny.fcmae_ft_in22k_in1k   ~24 GMAC        feat 768   <- xem tang kich thuoc co giup khong
                                      ~52 GMAC tong   VAN RE HON mot minh RAD-DINO (151)
```

Đọc ảnh mới là phần đắt, nên ba model dùng **chung** một lượt giải mã.
`in_chans=1` để `timm` tự gộp trọng số stem RGB thành 1 kênh — sạch hơn nhân bản 3 kênh.

## KHÔNG dùng checkpoint đã huấn luyện có giám sát trên MIMIC-CXR

Split test của CXR-LT là **tập con của MIMIC-CXR**. `torchxrayvision` `-all`/`-mimic_*`, CheXNet…
đã thấy ảnh test của ta **kèm nhãn** ⇒ rò rỉ trực tiếp. RAD-DINO được miễn vì nó **tự giám sát**
(DINOv2, không nhãn). ImageNet cũng an toàn — và còn cho đa dạng ở **cả hai** trục: kiến trúc **và**
miền pretrain. Một CNN pretrain trên CXR sẽ **tương quan cao hơn** với RAD-DINO, tức đi ngược mục tiêu.

## C0 · Config

In [1]:
import os, json, math, time, tarfile, io as _io, re, subprocess, sys
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    import timm
except ImportError:
    print('timm chua co -> cai...'); subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
    import timm
print(f'timm {timm.__version__}')

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES41 = os.path.join(OUT_DIR, 'stage4_nb41a')
RES49 = os.path.join(OUT_DIR, 'stage4_nb49')          # nguon MOC same-backbone
def ensure_res():
    '''Drive FUSE danh roi thu muc RONG — goi TRUOC moi cum ghi.'''
    os.makedirs(RES41, exist_ok=True); return RES41
ensure_res(); json.dump({'notebook': '41a'}, open(os.path.join(RES41, '_run_info.json'), 'w'))
LOCAL = '/content/cxrlt_518'

NAMES = ['convnextv2_nano.fcmae_ft_in22k_in1k',
         'densenet121.ra_in1k',
         'convnextv2_tiny.fcmae_ft_in22k_in1k']
SIZE, BATCH = 518, 64          # CNN o 518: stem stride-4 -> 129x129, activation LON. OOM thi ha 32.
SEED, SEEDS = 42, (42, 43, 44)
EPOCHS, LR, WD = 60, 3e-3, 1e-4        # y het `49` C2 — de so duoc voi probe cua RAD-DINO
NW = min(8, (os.cpu_count() or 2))
GATE_MAP = 0.15                        # probe >= 0.15 -> khoang cach mien nho, fine-tune dong duoc

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
YM = Y[LAB40].to_numpy('float32')
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
_rs = np.random.default_rng(SEED); _su = np.unique(SUB[TR])
_sel = _rs.choice(_su, size=int(round(0.15*len(_su))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL      # CUNG cach cat nhu 35/49
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
print(f'{len(ORDER):,} anh | khop {TF_.sum():,} | chon {SL.sum():,} | val {VA.sum():,}')
print(f'{len(NAMES)} backbone, MOT luot quet | {SIZE}px | ghi vao {RES41}')

timm 1.0.28
Mounted at /content/drive
377,110 anh | khop 219,785 | chon 39,086 | val 39,293
3 backbone, MOT luot quet | 518px | ghi vao /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage4_nb41a


## C1 · Nạp ba model + trích đặc trưng trong **một** lượt

`num_classes=0` ⇒ `timm` trả về đặc trưng **đã global-pool**. `in_chans=1` ⇒ `timm` gộp trọng số
stem RGB thành 1 kênh (tiết kiệm compute ở stem, và sạch hơn nhân bản 3 kênh như `31` làm cho ViT).

Mỗi model có `mean/std` riêng — lấy từ `resolve_model_data_config` của **chính model đó**, không
dùng chung của RAD-DINO. `memmap` + `_prog.json` nên đứt thì chạy lại cell này, tiếp từ chỗ dở.

In [3]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh local: {_n:,} / {len(ORDER):,}')
if _n < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')      # tu 28c C0
    assert os.path.isdir(SHARD_DIR), f'khong thay {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL, os.path.basename(nm))
        if os.path.exists(dst): return
        Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC) \
             .save(dst, 'JPEG', quality=95)
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()   # doc BULK; lap tren mount FUSE se TREO
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n:,} anh'

MODELS, MS = {}, {}
for nm in NAMES:
    m = timm.create_model(nm, pretrained=True, num_classes=0, in_chans=1).eval().to(dev)
    for p in m.parameters(): p.requires_grad_(False)
    cfg = timm.data.resolve_model_data_config(m)
    MODELS[nm] = m
    MS[nm] = (float(np.mean(cfg['mean'])), float(np.mean(cfg['std'])))   # anh xam -> TB 3 kenh
    print(f'  {nm:<40} feat {m.num_features:>5} | mean {MS[nm][0]:.3f} std {MS[nm][1]:.3f}')
# --- cong 1: KHONG model nao duoc huan luyen co giam sat tren MIMIC-CXR ---
_bad = [n for n in NAMES if re.search(r'xrv|torchxrayvision|mimic|chex|nih|rsna|padchest', n, re.I)]
assert not _bad, (f'{_bad} co ve la checkpoint hoc CO NHAN tren CXR. Split test cua CXR-LT la tap '
                  'con cua MIMIC-CXR -> RO RI. Chi dung ImageNet hoac tu giam sat.')
print(f'cong 1: ca {len(NAMES)} deu la ImageNet — khong ro ri nhan MIMIC-CXR')

class DSI(Dataset):
    def __init__(s, idx): s.idx = np.asarray(idx)
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = int(s.idx[k])
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        return torch.from_numpy(np.asarray(im, dtype='uint8')).unsqueeze(0), i

P_PROG = os.path.join(RES41, '_prog.json')
n0 = json.load(open(P_PROG))['n'] if os.path.exists(P_PROG) else 0
PF = {nm: os.path.join(RES41, f'feat_{nm.split(".")[0]}.npy') for nm in NAMES}
if n0 < len(ORDER):
    ensure_res()      # Drive FUSE danh roi thu muc trong luc stage anh (40-60 ph KHONG ghi gi
                      # vao Drive). Cung loi da dinh o 43/38c/38d — `ensure_res` phai goi TRUOC
                      # MOI cum ghi, khong chi mot lan o C0.
    FM = {nm: np.lib.format.open_memmap(
              PF[nm], mode=('r+' if os.path.exists(PF[nm]) else 'w+'), dtype='float32',
              shape=(len(ORDER), MODELS[nm].num_features)) for nm in NAMES}
    dl = DataLoader(DSI(np.arange(n0, len(ORDER))), batch_size=BATCH, shuffle=False,
                    num_workers=NW, pin_memory=True)
    t0, seen = time.time(), n0
    with torch.no_grad():
        for x, ii in tqdm(dl, desc='trich 3 CNN'):
            x = x.to(dev, non_blocking=True).float().div_(255)      # [B,1,H,W] — MOT lan giai ma
            for nm in NAMES:
                mu, sd = MS[nm]
                with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
                    f = MODELS[nm]((x - mu) / sd)
                FM[nm][ii.numpy()] = f.float().cpu().numpy()
            seen += len(ii)
            if seen % (BATCH*50) < BATCH:
                for nm in NAMES: FM[nm].flush()
                ensure_res(); json.dump({'n': int(seen)}, open(P_PROG, 'w'))
    for nm in NAMES: FM[nm].flush()
    ensure_res(); json.dump({'n': len(ORDER)}, open(P_PROG, 'w'))
    print(f'trich xong | {(time.time()-t0)/60:.1f} phut | '
          f'{(len(ORDER)-n0)/max(time.time()-t0,1e-9):.0f} img/s')
    del FM
for nm in NAMES: del MODELS[nm]
torch.cuda.empty_cache(); print('dac trung san sang')

anh local: 377,110 / 377,110
  convnextv2_nano.fcmae_ft_in22k_in1k      feat   640 | mean 0.449 std 0.226
  densenet121.ra_in1k                      feat  1024 | mean 0.449 std 0.226
  convnextv2_tiny.fcmae_ft_in22k_in1k      feat   768 | mean 0.449 std 0.226
cong 1: ca 3 deu la ImageNet — khong ro ri nhan MIMIC-CXR


/tmp/ipykernel_5471/3865563617.py:50: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  return torch.from_numpy(np.asarray(im, dtype='uint8')).unsqueeze(0), i


trich 3 CNN:   0%|          | 0/5893 [00:00<?, ?it/s]

/tmp/ipykernel_5471/3865563617.py:50: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  return torch.from_numpy(np.asarray(im, dtype='uint8')).unsqueeze(0), i
/tmp/ipykernel_5471/3865563617.py:50: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  return 

trich xong | 26.4 phut | 238 img/s
dac trung san sang


## C2 · Probe tuyến tính — **y hệt** `49` C2, để so được với RAD-DINO

In [4]:
class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/49. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        p = torch.sigmoid(x); pm = (p - s.clip).clamp(min=0)
        return -(y*torch.log(p.clamp(min=s.eps))*(1-p)**s.gp
                 + (1-y)*torch.log((1-pm).clamp(min=s.eps))*pm**s.gn).mean()

def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, sc = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-sc, kind='stable'); yy, ss = y[r], sc[r]
        tp, fp = np.cumsum(yy), np.cumsum(1-yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp/(tp+fp), tp/n1
        o[j] = float((np.diff(np.r_[0.0, rc])*pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}

def fit(X, seed):
    '''EPOCH CUOI, cosine trai tren EPOCHS, KHONG early-stop (nb35 §0.1).
       Chuan hoa bang thong ke cua LAT KHOP — khong cham lat chon, khong cham val.'''
    torch.manual_seed(seed)
    mu, sd = X[TF_].mean(0, keepdims=True), X[TF_].std(0, keepdims=True) + 1e-6
    z = lambda m: torch.from_numpy(((X[m]-mu)/sd).astype('float32')).to(dev)
    xt, xv = z(TF_), z(VA); yt = torch.from_numpy(YM[TF_]).to(dev)
    m = nn.Linear(xt.shape[1], C).to(dev); crit = ASL(gn=4, gp=0).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    bs = int(min(4096, max(256, len(xt)//64)))       # san 64 buoc/epoch (bay da dinh o nb35)
    for ep in range(EPOCHS):
        m.train(); perm = torch.randperm(len(xt), device=dev)
        for i in range(0, len(xt), bs):
            k = perm[i:i+bs]
            opt.zero_grad(set_to_none=True); crit(m(xt[k]), yt[k]).backward(); opt.step()
        sch.step()
    m.eval()
    with torch.no_grad(): return torch.sigmoid(m(xv)).cpu().numpy()

P_RES = os.path.join(RES41, 'probe.csv')
R = pd.read_csv(P_RES) if os.path.exists(P_RES) else pd.DataFrame()
for nm in NAMES:
    key = nm.split('.')[0]
    X = np.array(np.load(PF[nm], mmap_mode='r'))     # nap HAN, khong tra view cua memmap
    assert X.shape[0] == len(ORDER) and np.isfinite(X).all(), f'{key}: {X.shape}'
    assert np.abs(X).sum(1).min() > 0, f'{key}: co hang toan 0 -> trich chua xong'
    for s in SEEDS:
        if len(R) and ((R.model == key) & (R.seed == s)).any(): continue
        t0 = time.time(); pv = fit(X, s); ap = ap_cols(YM[VA], pv)
        R = pd.concat([R, pd.DataFrame([dict(model=key, seed=s, mAP=float(np.nanmean(ap)),
                                             **tiers(ap))])], ignore_index=True)
        ensure_res(); R.to_csv(P_RES, index=False)
        np.save(os.path.join(RES41, f'p_{key}_s{s}.npy'), pv)
        print(f'{key:<18} s{s}  mAP {R.mAP.iloc[-1]:.4f}  ({time.time()-t0:.0f}s)')
    del X
print('\nxong')

convnextv2_nano    s42  mAP 0.1496  (4s)
convnextv2_nano    s43  mAP 0.1489  (3s)
convnextv2_nano    s44  mAP 0.1491  (3s)
densenet121        s42  mAP 0.1395  (3s)
densenet121        s43  mAP 0.1396  (3s)
densenet121        s44  mAP 0.1399  (3s)
convnextv2_tiny    s42  mAP 0.1480  (3s)
convnextv2_tiny    s43  mAP 0.1481  (3s)
convnextv2_tiny    s44  mAP 0.1479  (3s)

xong


## C3 · `corr` — và **mốc same-backbone** lấy từ `49`

Spearman trên hạng, từng nhãn, trung bình 40 nhãn. AP là chỉ số **xếp hạng** nên tương quan phải đo
trên hạng, không trên xác suất thô.

Mốc: `corr` giữa **bốn arm ảnh của `49`** — cùng RAD-DINO pretrain, cùng đầu đọc, chỉ khác ảnh đầu
vào. Đó là mức đa dạng **cao nhất** mà một backbone duy nhất cho được.

In [5]:
from scipy.stats import rankdata
def corr(A, B):
    '''Spearman tung nhan roi TB 40 nhan.'''
    ra, rb = rankdata(A, axis=0), rankdata(B, axis=0)
    ra = (ra - ra.mean(0)) / (ra.std(0) + 1e-12); rb = (rb - rb.mean(0)) / (rb.std(0) + 1e-12)
    return float(np.mean((ra * rb).mean(0)))

_g49 = lambda a, s: os.path.join(RES49, f'p_{a}_s{s}.npy')
ARMS49 = [a for a in ('lb518', 'nosq518', 'crop518', 'lb672') if os.path.exists(_g49(a, SEEDS[0]))]
assert 'lb518' in ARMS49, (f'khong thay {_g49("lb518", SEEDS[0])} -> chay `49` truoc. '
                           'Khong co no thi khong co MOC de doc `corr`.')
P49 = {a: np.load(_g49(a, SEEDS[0])) for a in ARMS49}
BASE = [(a, b, corr(P49[a], P49[b])) for i, a in enumerate(ARMS49) for b in ARMS49[i+1:]]
MOC = min(c for _, _, c in BASE)
print('=== MOC SAME-BACKBONE (49, cung RAD-DINO pretrain, chi khac ANH dau vao) ===')
for a, b, c in sorted(BASE, key=lambda x: x[2]):
    print(f'  {a:<9} vs {b:<9} {c:.4f}')
print(f'  -> MOC = min = {MOC:.4f}  (da dang CAO NHAT ma MOT backbone cho duoc)')

RD = P49['lb518']       # RAD-DINO pretrain, lb518, cung dau doc — doi tuong so sanh
print(f'\n=== CNN vs RAD-DINO (s{SEEDS[0]}) ===')
piv = R.pivot_table(index='model', values=['mAP','head','medium','tail'], aggfunc='mean')
piv['bien_do'] = R.groupby('model').mAP.agg(lambda s: s.max()-s.min())
rows = []
for nm in NAMES:
    k = nm.split('.')[0]; f = os.path.join(RES41, f'p_{k}_s{SEEDS[0]}.npy')
    if not os.path.exists(f) or k not in piv.index: continue
    c = corr(np.load(f), RD)
    rows.append(dict(model=k, mAP=float(piv.loc[k, 'mAP']), tail=float(piv.loc[k, 'tail']),
                     corr_raddino=c, qua_mAP=piv.loc[k, 'mAP'] >= GATE_MAP, qua_corr=c < MOC))
T = pd.DataFrame(rows)
print(T.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
_rd = float(np.nanmean(ap_cols(YM[VA], RD)))
print(f'\n  RAD-DINO pretrain (lb518, cung dau doc): mAP {_rd:.4f}')

print('\n=== CONG ===')
print(f'  1  probe mAP >= {GATE_MAP:.2f}   -> khoang cach mien nho, fine-tune dong duoc')
print(f'  2  corr < {MOC:.4f}         -> co da dang THAT (thap hon moc same-backbone)')
if len(T) and not T.qua_corr.any():
    print(f'\n  !! KHONG con nao co corr < {MOC:.4f}.')
    print('     Kien truc CNN KHONG mang lai da dang o bai toan nay -> DONG huong ensemble CNN.')
    print('     Khoi fine-tune con nao (tiet kiem 41b). Ghi ket qua AM nay vao bao cao.')
elif len(T):
    _w = T[T.qua_corr & T.qua_mAP].sort_values('corr_raddino')
    if len(_w):
        print(f'\n  -> MOT con di tiep sang 41b: `{_w.model.iloc[0]}` '
              f'(mAP {_w.mAP.iloc[0]:.4f}, corr {_w.corr_raddino.iloc[0]:.4f})')
        print('     Cong cua 41b: CNN >= 0.85 x RAD-DINO don le VA ensemble - thanh vien manh nhat >= +0.005')
    else:
        print('\n  -> co con qua corr nhung KHONG con nao qua CA HAI cong. Xem bang tren.')
        print('     Mot mo hinh yeu-nhung-khac-biet va mot mo hinh manh-nhung-trung-lap deu vo dung.')
ensure_res(); T.to_csv(os.path.join(RES41, 'corr_table.csv'), index=False)

=== MOC SAME-BACKBONE (49, cung RAD-DINO pretrain, chi khac ANH dau vao) ===
  crop518   vs lb672     0.8236
  lb518     vs crop518   0.8331
  nosq518   vs crop518   0.8601
  nosq518   vs lb672     0.8827
  lb518     vs nosq518   0.8979
  lb518     vs lb672     0.9441
  -> MOC = min = 0.8236  (da dang CAO NHAT ma MOT backbone cho duoc)

=== CNN vs RAD-DINO (s42) ===
          model    mAP   tail  corr_raddino  qua_mAP  qua_corr
convnextv2_nano 0.1492 0.0255        0.5193    False      True
    densenet121 0.1397 0.0157        0.4683    False      True
convnextv2_tiny 0.1480 0.0220        0.5222    False      True

  RAD-DINO pretrain (lb518, cung dau doc): mAP 0.2042

=== CONG ===
  1  probe mAP >= 0.15   -> khoang cach mien nho, fine-tune dong duoc
  2  corr < 0.8236         -> co da dang THAT (thap hon moc same-backbone)

  -> co con qua corr nhung KHONG con nao qua CA HAI cong. Xem bang tren.
     Mot mo hinh yeu-nhung-khac-biet va mot mo hinh manh-nhung-trung-lap deu vo dung.


## C4 · Ensemble **miễn phí** ngay tại đây — probe + probe, chưa fine-tune

In [6]:
# Ensemble o muc PROBE khong phai ket qua cuoi (ca hai deu la probe tuyen tinh tren dac trung dong
# cung), nhung no cho biet HUONG: neu ngay ca o muc nay ensemble cung khong nhac gi thi 41b khong
# co ly do ton tai. Dung TRUNG BINH HANG song song voi trung binh xac suat — AP la chi so XEP HANG
# va hai mo hinh huan luyen rieng rat co the lech calibration.
rk = lambda P: rankdata(P, axis=0) / len(P)
print(f'{"to hop":<44} {"prob":>8} {"rank":>8} {"vs tot nhat":>12}')
_best_solo = _rd
for nm in NAMES:
    k = nm.split('.')[0]; f = os.path.join(RES41, f'p_{k}_s{SEEDS[0]}.npy')
    if not os.path.exists(f): continue
    P = np.load(f); _best_solo = max(_best_solo, float(np.nanmean(ap_cols(YM[VA], P))))
    mp = float(np.nanmean(ap_cols(YM[VA], (P + RD) / 2)))
    mr = float(np.nanmean(ap_cols(YM[VA], (rk(P) + rk(RD)) / 2)))
    print(f'{k + " + raddino":<44} {mp:>8.4f} {mr:>8.4f} '
          f'{max(mp, mr) - max(float(np.nanmean(ap_cols(YM[VA], P))), _rd):>+12.4f}')
_all = [np.load(os.path.join(RES41, f'p_{n.split(".")[0]}_s{SEEDS[0]}.npy')) for n in NAMES
        if os.path.exists(os.path.join(RES41, f'p_{n.split(".")[0]}_s{SEEDS[0]}.npy'))] + [RD]
if len(_all) > 2:
    mp = float(np.nanmean(ap_cols(YM[VA], np.mean(_all, 0))))
    mr = float(np.nanmean(ap_cols(YM[VA], np.mean([rk(p) for p in _all], 0))))
    print(f'{"TAT CA " + str(len(_all)):<44} {mp:>8.4f} {mr:>8.4f} {max(mp,mr)-_best_solo:>+12.4f}')
print(f'\n  tot nhat don le: {_best_solo:.4f} | moc 17 (ensemble 3 ckpt CUNG mo hinh): macro +0.00112')
print('  Trong so DEU. Chon trong so tren val roi bao cao tren val la dung bay §13 cua ke hoach.')

to hop                                           prob     rank  vs tot nhat
convnextv2_nano + raddino                      0.2006   0.1852      -0.0035
densenet121 + raddino                          0.1959   0.1803      -0.0083
convnextv2_tiny + raddino                      0.1990   0.1841      -0.0052
TAT CA 4                                       0.1878   0.1768      -0.0164

  tot nhat don le: 0.2042 | moc 17 (ensemble 3 ckpt CUNG mo hinh): macro +0.00112
  Trong so DEU. Chon trong so tren val roi bao cao tren val la dung bay §13 cua ke hoach.


## C5 · Ghi `REPORT_STAGE4_NB41A.md`

In [7]:
L = []; w = L.append
w('# REPORT · `nb41a` — ba CNN có mang lại đa dạng cho ensemble không'); w('')
w(f'`asl40 E={EPOCHS}` epoch cuối · seeds {list(SEEDS)} · probe tuyến tính trên đặc trưng CNN '
  f'**đông cứng** (ImageNet) · val {int(VA.sum()):,} ảnh · `{SIZE}px` · một lượt quét cho cả ba.')
w('')
w('| model | mAP | biên độ | tail | `corr` với RAD-DINO | qua cổng mAP | qua cổng corr |')
w('|---|---|---|---|---|---|---|')
for r in T.itertuples():
    w(f'| `{r.model}` | {r.mAP:.4f} | {piv.loc[r.model, "bien_do"]:.4f} | {r.tail:.4f} | '
      f'{r.corr_raddino:.4f} | {"có" if r.qua_mAP else "KHÔNG"} | {"có" if r.qua_corr else "KHÔNG"} |')
w(f'| `raddino` (mốc) | {_rd:.4f} | — | — | — | — | — |')
w(''); w('```')
w('MOC same-backbone (49, cung RAD-DINO pretrain, chi khac anh dau vao):')
for a, b, c in sorted(BASE, key=lambda x: x[2]): w(f'  {a:<9} vs {b:<9} {c:.4f}')
w(f'MOC = min = {MOC:.4f}'); w('```'); w('')
w('**Mốc `corr` lấy từ dữ liệu, không phỏng đoán.** Bốn arm ảnh của `49` dùng chung một backbone '
  'RAD-DINO pretrain và cùng đầu đọc; `corr` giữa chúng là mức đa dạng **cao nhất** mà một backbone '
  'cho được. CNN không xuống dưới mức đó thì nó không mang lại gì mà một arm ảnh khác không mang lại.')
w(''); w('## Những gì kết quả này KHÔNG trả lời'); w('')
w('- **Đây là probe trên đặc trưng đông cứng**, không phải fine-tune. `04b` đo LP `0.3191` → FT '
  '`0.4201` (`+0.101`) trên cohort cũ ⇒ thứ hạng có thể đổi sau fine-tune. Cổng này chỉ loại các '
  'ứng viên **rõ ràng** không đa dạng.')
w('- **ImageNet, không phải CXR.** Cố ý: một CNN pretrain trên CXR chia sẻ cùng miền với RAD-DINO ⇒ '
  '`corr` **cao hơn** ⇒ ensemble mua được **ít hơn**. Và mọi checkpoint học có nhãn trên MIMIC-CXR '
  'đều **rò rỉ** (test CXR-LT là tập con của MIMIC-CXR).')
w('- **Trọng số đều.** Chọn trọng số per-label trên val rồi báo cáo trên val là bẫy `§13`; với '
  '19–129 ca dương ở tail thì nó overfit chắc chắn.')
TXT = '\n'.join(L); ensure_res()
open(os.path.join(RES41, 'REPORT_STAGE4_NB41A.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)

# REPORT · `nb41a` — ba CNN có mang lại đa dạng cho ensemble không

`asl40 E=60` epoch cuối · seeds [42, 43, 44] · probe tuyến tính trên đặc trưng CNN **đông cứng** (ImageNet) · val 39,293 ảnh · `518px` · một lượt quét cho cả ba.

| model | mAP | biên độ | tail | `corr` với RAD-DINO | qua cổng mAP | qua cổng corr |
|---|---|---|---|---|---|---|
| `convnextv2_nano` | 0.1492 | 0.0008 | 0.0255 | 0.5193 | KHÔNG | có |
| `densenet121` | 0.1397 | 0.0003 | 0.0157 | 0.4683 | KHÔNG | có |
| `convnextv2_tiny` | 0.1480 | 0.0002 | 0.0220 | 0.5222 | KHÔNG | có |
| `raddino` (mốc) | 0.2042 | — | — | — | — | — |

```
MOC same-backbone (49, cung RAD-DINO pretrain, chi khac anh dau vao):
  crop518   vs lb672     0.8236
  lb518     vs crop518   0.8331
  nosq518   vs crop518   0.8601
  nosq518   vs lb672     0.8827
  lb518     vs nosq518   0.8979
  lb518     vs lb672     0.9441
MOC = min = 0.8236
```

**Mốc `corr` lấy từ dữ liệu, không phỏng đoán.** Bốn arm ảnh của `49` dùng chung một backbone RAD-DINO p